# Stage 10 — XGBoost Extension

This notebook evaluates XGBoost on the project's existing binary pneumonia classification task. **XGBoost is an extension experiment and is not part of the original reference paper.** It uses the already-fixed Stage 2 split and shared preprocessing; previous models are evaluated from their saved artifacts, never retrained here by default.

## 1. Objective and reference models

The reference experiments comprise Logistic Regression, SVM, Random Forest, and the custom 10-layer CNN. XGBoost is an additional tree-boosting experiment; it is not attributed to the paper. Classical reference models use flattened normalized pixels, while the CNN learns spatial features from the same 128 × 128 images.

In [ ]:
import csv
import json
import sys
from pathlib import Path
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd()
if not (ROOT / "metadata").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "metadata").is_dir():
    raise FileNotFoundError("Could not locate the project root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULT_PATH = ROOT / "results" / "xgboost_results.json"
COMPARISON_PATH = ROOT / "results" / "model_comparison.csv"
print("Project root:", ROOT)

## 2. Why XGBoost?

Tree boosting provides a different classical machine-learning approach. This experiment tests whether nonlinear boosting on PCA-compressed image vectors is competitive with existing baselines. It is not assumed to outperform the CNN or any other model.

## 3. Data preparation

The shared loader decodes the original DICOMs, resizes and normalizes them using the saved training-only pixel statistics, then flattens each image. The fixed `metadata/split_metadata.csv` assignments are reused unchanged:

**128 × 128 grayscale → shared normalization → 16,384 flattened pixels → train-only PCA → XGBoost**

Temporary memory-mapped feature matrices limit RAM use; no preprocessed image copies are retained.

## 4. PCA

Randomized-SVD PCA reduces the 16,384 pixel features to **256 components**. PCA is fit only on `X_train`; validation and test data are transformed with the fitted train PCA. The randomized solver, one power iteration, and a seed of 42 make the decomposition practical and deterministic without using validation/test information to estimate components.

In [ ]:
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    from scripts.train_xgboost import run_smoke_test
    run_smoke_test(ROOT)

RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    from scripts.train_xgboost import run_experiment
    run_experiment(ROOT)
if not RESULT_PATH.is_file() or not COMPARISON_PATH.is_file():
    raise FileNotFoundError(
        "Run `python scripts/train_xgboost.py` once to create the saved experiment results."
    )
xgb_result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
pca = xgb_result["pca_configuration"]
parameters = xgb_result["xgboost_parameters"]
display(Markdown(
    f"**Input features:** {xgb_result['original_feature_dimension']:,}  \n"
    f"**PCA components:** {xgb_result['pca_feature_dimension']}  \n"
    f"**Explained variance:** {pca['total_explained_variance_ratio']:.2%}  \n"
    f"**PCA fit samples:** {pca['fit_sample_count']:,} (train only)  \n"
    f"**Split SHA-256:** `{xgb_result['dataset']['split_metadata_sha256']}`"
))
print("XGBoost parameters:", json.dumps(parameters, indent=2))
print("PCA configuration:", json.dumps(pca, indent=2))

## 5. Training configuration and execution

The default notebook path reads the already-produced result and does not train again. The 500-train/100-validation smoke test can be enabled separately with `RUN_SMOKE_TEST = True`. To launch the one full run interactively, set `RUN_EXPERIMENT = True` above (or run `python scripts/train_xgboost.py` once). XGBoost uses `binary:logistic`, log-loss, 100 trees, depth 4, learning rate 0.05, row/column subsampling 0.8, seed 42, and histogram tree building. Validation metrics are reported but are not used for early stopping or model selection.

## 6. Evaluation

The tables below are populated from the saved experiment JSON. Confusion matrices use rows=true labels and columns=predicted labels, in Healthy (0), Pneumonia (1) order. The test set is used only for final evaluation.

In [ ]:
def metrics_table(results):
    headers = ["Split", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    rows = []
    for split, values in results.items():
        rows.append(
            f"| {split.title()} | {values['accuracy']:.4f} | {values['precision']:.4f} | "
            f"{values['recall']:.4f} | {values['f1']:.4f} | {values.get('roc_auc', float('nan')):.4f} |"
        )
    return "| " + " | ".join(headers) + " |\n|---|---:|---:|---:|---:|---:|\n" + "\n".join(rows)

display(Markdown(metrics_table({
    "train": xgb_result["train_metrics"],
    "validation": xgb_result["validation_metrics"],
    "test": xgb_result["test_metrics"],
})))
for split in ("train", "validation", "test"):
    print(f"{split.title()} confusion matrix:", xgb_result[f"{split}_metrics"]["confusion_matrix"])
print(f"PCA fit time: {xgb_result['pca_fit_time_seconds']:.2f}s")
print(f"XGBoost fit time: {xgb_result['xgboost_fit_time_seconds']:.2f}s")

In [ ]:
curve = xgb_result["test_metrics"].get("roc_curve")
if curve:
    width, height, margin = 420, 420, 44
    points = " ".join(
        f"{margin + fpr * (width - 2 * margin):.1f},{height - margin - tpr * (height - 2 * margin):.1f}"
        for fpr, tpr in zip(curve["false_positive_rate"], curve["true_positive_rate"])
    )
    display(SVG(
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">'
        f'<rect width="100%" height="100%" fill="white"/>'
        f'<line x1="{margin}" y1="{height-margin}" x2="{width-margin}" y2="{margin}" stroke="#aaa" stroke-dasharray="5,5"/>'
        f'<polyline points="{points}" fill="none" stroke="#1769aa" stroke-width="3"/>'
        f'<text x="{width/2}" y="{height-8}" text-anchor="middle">False-positive rate</text>'
        f'<text x="15" y="{height/2}" transform="rotate(-90 15 {height/2})" text-anchor="middle">True-positive rate</text>'
        "</svg>"
    ))
else:
    print("ROC curve data is not available.")

## 7. Comparison with existing models

Logistic Regression, SVM, and Random Forest values below come from their saved model artifacts evaluated on the unchanged test split; none were retrained. The CNN row uses inference from the saved Stage 8 checkpoint (checkpoint epoch shown in the result JSON). The comparison therefore includes actual accuracy, F1, precision, and recall for each model.

In [ ]:
with COMPARISON_PATH.open(newline="", encoding="utf-8-sig") as f:
    comparison_rows = list(csv.DictReader(f))
comparison_headers = ["Model", "Representation", "Test Accuracy", "Test F1", "Test Precision", "Test Recall"]
comparison_markdown = "| " + " | ".join(comparison_headers) + " |\n|---|---|---:|---:|---:|---:|\n"
comparison_markdown += "\n".join(
    f"| {row['model']} | {row['feature_representation']} | {float(row['test_accuracy']):.4f} | "
    f"{float(row['test_f1']):.4f} | {float(row['test_precision']):.4f} | {float(row['test_recall']):.4f} |"
    for row in comparison_rows
)
display(Markdown(comparison_markdown))
print("CNN comparison checkpoint epoch:", xgb_result["cnn_comparison_checkpoint_epoch"])

## 8. Discussion

Interpret the comparison from the measured test metrics, not from model-family assumptions. PCA may discard information while reducing the input from 16,384 features to 256; the recorded explained variance quantifies retained pixel variance, not class information. Randomized PCA and temporary memory-mapped feature matrices reduce decomposition overhead and memory pressure, but image decoding and PCA still add runtime. CNNs can retain an advantage by learning local and spatial feature hierarchies, while the PCA representation supplies XGBoost with global linear projections only. The local labeled cohort (14,863 images) differs from the paper's reported 17,489, and these results should be treated as a fixed-split local experiment.

## 9. Extension conclusion

**XGBoost was evaluated as an extension to the reference methodology and was not part of the original paper.** The experiment uses the existing fixed split, fits PCA on training data only, and evaluates the selected fixed configuration without a hyperparameter search.